In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def compare_models(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=random_state
    )

    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    scale_pos_weight = neg / pos   # XGBoost는 class_weight 대신 이걸 씀

    models = {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=1000),
        "RandomForest": RandomForestClassifier(class_weight="balanced", n_estimators=300, random_state=random_state),
        "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=random_state),
        "LightGBM": LGBMClassifier(class_weight="balanced", random_state=random_state, verbose=-1),
    }

    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        results.append({"model": name, "roc_auc": auc})

    return pd.DataFrame(results).sort_values("roc_auc", ascending=False)


In [19]:
tox21 = pd.read_csv("./data/hazard/moleculenet_tox21.csv")
fp = np.load("./data/processed/moleculenet_tox21_fingerprint.npy")

task = "NR-AR"
mask = tox21[task].notna()          # 그 태스크에 라벨 있는 행만
X = fp[mask.values]
y = tox21.loc[mask, task].values

print(compare_models(X, y))

                model   roc_auc
2        RandomForest  0.752736
1  LogisticRegression  0.736764
4            LightGBM  0.725870
3             XGBoost  0.724838
0               Dummy  0.500000


In [20]:
label_cols = [c for c in tox21.columns if c not in ("SMILES", "InChIKey")]

all_results = []
for task in label_cols:
    mask = tox21[task].notna()
    X = fp[mask.values]
    y = tox21.loc[mask, task].values
    res = compare_models(X, y)
    res["task"] = task
    all_results.append(res)

summary = pd.concat(all_results)
print(summary.pivot(index="model", columns="task", values="roc_auc"))

task                   NR-AR  NR-AR-LBD    NR-AhR  NR-Aromatase     NR-ER  \
model                                                                       
Dummy               0.500000   0.500000  0.500000      0.500000  0.500000   
LightGBM            0.725870   0.781446  0.899320      0.822391  0.679330   
LogisticRegression  0.736764   0.799380  0.872214      0.839857  0.677944   
RandomForest        0.752736   0.821309  0.907437      0.870083  0.702982   
XGBoost             0.724838   0.802291  0.880994      0.817971  0.657547   

task                NR-ER-LBD  NR-PPAR-gamma    SR-ARE  SR-ATAD5    SR-HSE  \
model                                                                        
Dummy                0.500000       0.500000  0.500000  0.500000  0.500000   
LightGBM             0.771126       0.770734  0.808549  0.842172  0.752686   
LogisticRegression   0.793620       0.760165  0.771046  0.843502  0.721051   
RandomForest         0.790175       0.805569  0.823170  0.893948  0.81

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

# fingerprint는 이미 0/1 숫자 배열이라 결측치·범주형 처리가 필요 없음
# 트리 계열은 스케일에 영향 안 받아서 그대로 통과
# 선형 계열(Ridge)은 alpha 규제가 피처 스케일에 민감해서 표준화 적용
preprocess_for_tree = 'passthrough'
preprocess_for_linear = StandardScaler()

# alpha: alpha가 커질수록 규제가 강해짐. alpha가 작아지면 LinearRegression과 비슷
models = {
    'Dummy': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', DummyRegressor(strategy='mean'))
    ]),
    'LinearRegression': Pipeline([
        ('preprocess', preprocess_for_linear),
        ('model', LinearRegression())
    ]),
    'Ridge': Pipeline([
        ('preprocess', preprocess_for_linear),
        ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))
    ]),
    'DecisionTree': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))
    ]),
    'RandomForest': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=300, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', LGBMRegressor(random_state=RANDOM_STATE, verbose=-1))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))
    ]),
    'HistGradientBoosting': Pipeline([
        ('preprocess', preprocess_for_tree),
        ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE))
    ])
}

def evaluate_regressors(X, y, models, random_state=RANDOM_STATE):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    results = []
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        results.append({
            "model": name,
            "R2": r2_score(y_test, y_pred),
            "RMSE": root_mean_squared_error(y_test, y_pred),
            "MAE": mean_absolute_error(y_test, y_pred),
        })
    return pd.DataFrame(results).sort_values("R2", ascending=False)

In [15]:
mp = pd.read_csv("./data/processed/crc_mp_c.csv")
mp_fp = np.load("./data/processed/crc_mp_c_fingerprint.npy")
X = mp_fp
y = mp["Processed tmp/ºC"].values
print(evaluate_regressors(X, y, models))

                  model        R2       RMSE        MAE
8  HistGradientBoosting  0.631928  59.628855  43.970805
6              LightGBM  0.631928  59.628855  43.970805
5               XGBoost  0.602422  61.972782  46.011676
4          RandomForest  0.589605  62.963822  45.442690
7      GradientBoosting  0.535161  67.010216  51.151215
3          DecisionTree  0.371010  77.949242  56.673045
2                 Ridge  0.311574  81.548985  61.578376
1      LinearRegression  0.310704  81.600482  61.615990
0                 Dummy -0.004921  98.527151  77.194001


In [10]:
bp = pd.read_csv("./data/processed/crc_bp_c.csv")
bp_fp = np.load("./data/processed/crc_bp_c_fingerprint.npy")

X = bp_fp
y = bp["Processed tbp/ºC"].values

print(evaluate_regressors(X, y, models))

                  model        R2       RMSE        MAE
8  HistGradientBoosting  0.721252  46.806370  33.951798
6              LightGBM  0.721252  46.806370  33.951798
5               XGBoost  0.703717  48.256133  35.877568
4          RandomForest  0.671412  50.818878  37.065278
7      GradientBoosting  0.651084  52.367302  39.756260
3          DecisionTree  0.483862  63.691611  47.715636
2                 Ridge  0.422460  67.373728  50.084998
1      LinearRegression  0.418310  67.615320  50.246810
0                 Dummy -0.000007  88.654549  69.825929


In [13]:
# 1. 두 파이프라인이 실제로 다른 객체인지
print(models['LightGBM'] is models['HistGradientBoosting'])

# 2. 내부 모델 클래스가 진짜 다른지
print(type(models['LightGBM'].named_steps['model']))
print(type(models['HistGradientBoosting'].named_steps['model']))

# 3. lightgbm이 제대로 import됐는지 (설치 문제로 조용히 실패했을 가능성 체크)
import lightgbm
print(lightgbm.__version__)

False
<class 'lightgbm.sklearn.LGBMRegressor'>
<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingRegressor'>
4.7.0


In [22]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

def get_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * 6
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
    ]

# mp 데이터셋의 SMILES마다 서술자 6개씩 계산
desc_list = [get_descriptors(s) for s in mp["SMILES"]]
desc_array = np.array(desc_list)
print("서술자 배열 크기:", desc_array.shape)   # (7037, 6) 나와야 정상

# 기존 fingetprint(2024차원)에 서술자(6차원)를 옆으로 이어붙이기
mp_fp = np.load("./data/processed/crc_mp_c_fingerprint.npy")
X_extended = np.hstack([mp_fp, desc_array])
print("확장된 X 크기:", X_extended.shape)   # (7037, 2054) 나와야 정상

y = mp["Processed tmp/ºC"].values

print(evaluate_regressors(X_extended, y, models))

서술자 배열 크기: (7037, 6)
확장된 X 크기: (7037, 2054)
                  model        R2       RMSE        MAE
8  HistGradientBoosting  0.721603  51.858743  36.890303
6              LightGBM  0.720776  51.935755  37.162292
5               XGBoost  0.696209  54.172338  38.406702
4          RandomForest  0.689978  54.725021  38.358046
7      GradientBoosting  0.660781  57.244008  41.823730
3          DecisionTree  0.460119  72.216918  50.883410
2                 Ridge  0.377359  77.554786  58.178608
1      LinearRegression  0.376618  77.600931  58.206472
0                 Dummy -0.004921  98.527151  77.194001


In [23]:
# bp 데이터셋의 SMILES마다 서술자 6개씩 계산 (mp 때와 동일한 함수 재사용)
desc_list_bp = [get_descriptors(s) for s in bp["SMILES"]]
desc_array_bp = np.array(desc_list_bp)
print("서술자 배열 크기:", desc_array_bp.shape)   # (4795, 6) 나와야 정상

# 기존 fingerprint(2048차원)에 서술자(6차원) 이어붙이기
bp_fp = np.load("./data/processed/crc_bp_c_fingerprint.npy")
X_extended_bp = np.hstack([bp_fp, desc_array_bp])
print("확장된 X 크기:", X_extended_bp.shape)   # (4795, 2054) 나와야 정상

y_bp = bp["Processed tbp/ºC"].values

print(evaluate_regressors(X_extended_bp, y_bp, models))

서술자 배열 크기: (4795, 6)
확장된 X 크기: (4795, 2054)
                  model        R2       RMSE        MAE
6              LightGBM  0.908172  26.865059  16.920819
8  HistGradientBoosting  0.907874  26.908633  16.703275
5               XGBoost  0.900609  27.949364  18.053536
4          RandomForest  0.884285  30.157388  19.195520
7      GradientBoosting  0.868273  32.176366  22.369583
3          DecisionTree  0.791779  40.454065  25.530619
2                 Ridge  0.698006  48.719002  35.632730
1      LinearRegression  0.695298  48.937008  35.776198
0                 Dummy -0.000007  88.654549  69.825929


In [24]:
print("bp 중복 InChIKey 개수:", bp["InChIKey"].duplicated().sum())
print("mp 중복 InChIKey 개수:", mp["InChIKey"].duplicated().sum())

bp 중복 InChIKey 개수: 0
mp 중복 InChIKey 개수: 0


In [25]:
# tox21 데이터셋의 SMILES마다 서술자 6개씩 계산 (같은 함수 재사용)
desc_list_tox21 = [get_descriptors(s) for s in tox21["SMILES"]]
desc_array_tox21 = np.array(desc_list_tox21)
print("서술자 배열 크기:", desc_array_tox21.shape)   # (7831, 6) 나와야 정상

# 기존 fingerprint(2048차원)에 서술자(6차원) 이어붙이기
tox21_fp = np.load("./data/processed/moleculenet_tox21_fingerprint.npy")
X_extended_tox21 = np.hstack([tox21_fp, desc_array_tox21])

# 태스크 하나(예: NR-AR)에 적용
task = "NR-AR"
mask = tox21[task].notna()
X = X_extended_tox21[mask.values]
y = tox21.loc[mask, task].values

print(compare_models(X, y))

서술자 배열 크기: (7831, 6)


                model   roc_auc
2        RandomForest  0.761201
4            LightGBM  0.749530
3             XGBoost  0.737558
1  LogisticRegression  0.734120
0               Dummy  0.500000


In [26]:
label_cols = [c for c in tox21.columns if c not in ("SMILES", "InChIKey")]

all_results = []
for task in label_cols:
    mask = tox21[task].notna()
    X = X_extended_tox21[mask.values]
    y = tox21.loc[mask, task].values
    res = compare_models(X, y)
    res["task"] = task
    all_results.append(res)

summary = pd.concat(all_results)
print(summary.pivot(index="model", columns="task", values="roc_auc"))

task                   NR-AR  NR-AR-LBD    NR-AhR  NR-Aromatase     NR-ER  \
model                                                                       
Dummy               0.500000   0.500000  0.500000      0.500000  0.500000   
LightGBM            0.749530   0.775528  0.891481      0.856410  0.694666   
LogisticRegression  0.734120   0.794196  0.876163      0.847436  0.678640   
RandomForest        0.761201   0.809733  0.913290      0.892670  0.723958   
XGBoost             0.737558   0.803538  0.892632      0.863937  0.690607   

task                NR-ER-LBD  NR-PPAR-gamma    SR-ARE  SR-ATAD5    SR-HSE  \
model                                                                        
Dummy                0.500000       0.500000  0.500000  0.500000  0.500000   
LightGBM             0.824905       0.836393  0.831610  0.882484  0.767352   
LogisticRegression   0.801925       0.769936  0.779497  0.848142  0.743642   
RandomForest         0.812783       0.871552  0.859885  0.924563  0.84

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

mp = pd.read_csv('data/processed/crc_mp_c.csv')
mp_fp = np.load('data/processed/crc_mp_c_fingerprint.npy')
y = mp['Processed tmp/ºC'].values

# 217개 서술자 전체 계산
desc_names = None
all_desc = []
for smi in mp['SMILES']:
    m = Chem.MolFromSmiles(smi)
    vals = Descriptors.CalcMolDescriptors(m)
    if desc_names is None:
        desc_names = list(vals.keys())
    all_desc.append(list(vals.values()))
all_desc = np.nan_to_num(np.array(all_desc, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

X_full = np.hstack([mp_fp, all_desc])
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

FP_DIM = mp_fp.shape[1]  # 2048, 서술자는 이 뒤 컬럼부터

def permutation_importance_subset(model, X_test, y_test, start_idx, feature_names, n_repeats=5, random_state=42):
    """서술자 컬럼만 하나씩 무작위로 섞어서, 섞기 전후 R² 하락폭을 중요도로 삼는다.
    HistGradientBoostingRegressor는 feature_importances_를 지원하지 않아 직접 구현."""
    rng = np.random.RandomState(random_state)
    baseline = r2_score(y_test, model.predict(X_test))
    rows = []
    for offset, name in enumerate(feature_names):
        col = start_idx + offset
        drops = []
        for _ in range(n_repeats):
            X_perm = X_test.copy()
            rng.shuffle(X_perm[:, col])
            score = r2_score(y_test, model.predict(X_perm))
            drops.append(baseline - score)
        rows.append({'descriptor': name, 'importance_mean': np.mean(drops), 'importance_std': np.std(drops)})
    return pd.DataFrame(rows).sort_values('importance_mean', ascending=False)

importance_df = permutation_importance_subset(model, X_test, y_test, FP_DIM, desc_names, n_repeats=5)
print(importance_df.head(20).to_string(index=False))
print('기여 없음(중요도 ≤ 0):', (importance_df['importance_mean'] <= 0).sum(), '/', len(importance_df))

       descriptor  importance_mean  importance_std
        NHOHCount         0.076744        0.005112
             TPSA         0.068986        0.008030
          BertzCT         0.066922        0.002706
        RingCount         0.024748        0.001801
       NumHDonors         0.015922        0.001704
 FpDensityMorgan3         0.011196        0.001776
NumRotatableBonds         0.011126        0.001387
         BalabanJ         0.009411        0.001368
   NumHeteroatoms         0.008250        0.001730
          NOCount         0.005411        0.001604
       SlogP_VSA1         0.004973        0.001976
        LabuteASA         0.004232        0.005007
      fr_bicyclic         0.003962        0.001646
       SlogP_VSA2         0.003718        0.000412
 MaxPartialCharge         0.003641        0.000511
         SMR_VSA5         0.003612        0.001885
      BCUT2D_MRHI         0.003576        0.001190
 MinPartialCharge         0.003424        0.001192
      VSA_EState1         0.003

In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

mp = pd.read_csv('data/processed/crc_mp_c.csv')
mp_fp = np.load('data/processed/crc_mp_c_fingerprint.npy')
y = mp['Processed tmp/ºC'].values

# 217개 서술자 전체 계산 (한 번만)
desc_names = None
all_desc = []
for smi in mp['SMILES']:
    m = Chem.MolFromSmiles(smi)
    vals = Descriptors.CalcMolDescriptors(m)
    if desc_names is None:
        desc_names = list(vals.keys())
    all_desc.append(list(vals.values()))
all_desc = np.nan_to_num(np.array(all_desc, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

def permutation_importance_train_only(model, X_train, y_train, desc_start, desc_names, n_repeats=3, random_state=42):
    """중요도는 train으로만 계산 (test는 절대 안 건드림 -> 누수 방지)"""
    rng = np.random.RandomState(random_state)
    baseline = r2_score(y_train, model.predict(X_train))
    rows = []
    for offset, name in enumerate(desc_names):
        col = desc_start + offset
        drops = []
        for _ in range(n_repeats):
            X_perm = X_train.copy()
            rng.shuffle(X_perm[:, col])
            drops.append(baseline - r2_score(y_train, model.predict(X_perm)))
        rows.append({'descriptor': name, 'importance_mean': np.mean(drops)})
    return pd.DataFrame(rows).sort_values('importance_mean', ascending=False)

N_CANDIDATES = [0, 5, 10, 15, 20, 25, 30, 40, 60, 100, 217]
FP_DIM = mp_fp.shape[1]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_results = {n: [] for n in N_CANDIDATES}

for fold_i, (train_idx, test_idx) in enumerate(kf.split(mp_fp)):
    fp_train, fp_test = mp_fp[train_idx], mp_fp[test_idx]
    desc_train, desc_test = all_desc[train_idx], all_desc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_full_train = np.hstack([fp_train, desc_train])

    # 1) 이 fold의 train으로만 전체 모델 학습 -> 중요도 계산 (test 미사용)
    full_model = HistGradientBoostingRegressor(random_state=42)
    full_model.fit(X_full_train, y_train)
    importance_df = permutation_importance_train_only(full_model, X_full_train, y_train, FP_DIM, desc_names)

    # 2) 상위 N개씩 골라서 재학습, test는 여기서 딱 한 번만 사용
    for n in N_CANDIDATES:
        top_names = importance_df.head(n)['descriptor'].tolist() if n > 0 else []
        idxs = [desc_names.index(nm) for nm in top_names]
        X_tr = np.hstack([fp_train, desc_train[:, idxs]]) if n > 0 else fp_train
        X_te = np.hstack([fp_test, desc_test[:, idxs]]) if n > 0 else fp_test
        m = HistGradientBoostingRegressor(random_state=42)
        m.fit(X_tr, y_train)
        fold_results[n].append(r2_score(y_test, m.predict(X_te)))
    print(f'fold {fold_i+1}/5 완료')

print()
print(f"{'N':>5} {'R2 평균':>10} {'R2 표준편차':>10}")
for n in N_CANDIDATES:
    scores = fold_results[n]
    print(f'{n:>5} {np.mean(scores):>10.4f} {np.std(scores):>10.4f}')

fold 1/5 완료
fold 2/5 완료
fold 3/5 완료
fold 4/5 완료
fold 5/5 완료

    N      R2 평균    R2 표준편차
    0     0.6466     0.0110
    5     0.7319     0.0140
   10     0.7443     0.0110
   15     0.7517     0.0083
   20     0.7524     0.0129
   25     0.7579     0.0108
   30     0.7586     0.0128
   40     0.7623     0.0117
   60     0.7637     0.0110
  100     0.7629     0.0128
  217     0.7640     0.0132


In [4]:
import pandas as pd
import numpy as np
import time
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score

N_CANDIDATES = [0, 5, 10, 15, 20, 25, 30, 40, 60, 100, 217]


def compute_all_descriptors(smiles_list):
    desc_names = None
    all_desc = []
    for smi in smiles_list:
        m = Chem.MolFromSmiles(smi)
        vals = Descriptors.CalcMolDescriptors(m)
        if desc_names is None:
            desc_names = list(vals.keys())
        all_desc.append(list(vals.values()))
    all_desc = np.nan_to_num(np.array(all_desc, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    all_desc = np.clip(all_desc, -1e6, 1e6)
    return desc_names, all_desc


def permutation_importance_train_only(model, X_train, y_train, desc_start, desc_names, metric_fn, n_repeats=3, random_state=42):
    rng = np.random.RandomState(random_state)
    baseline = metric_fn(y_train, model.predict(X_train)) if metric_fn is r2_score else metric_fn(y_train, model.predict_proba(X_train)[:, 1])
    rows = []
    for offset, name in enumerate(desc_names):
        col = desc_start + offset
        drops = []
        for _ in range(n_repeats):
            Xp = X_train.copy()
            rng.shuffle(Xp[:, col])
            score = metric_fn(y_train, model.predict(Xp)) if metric_fn is r2_score else metric_fn(y_train, model.predict_proba(Xp)[:, 1])
            drops.append(baseline - score)
        rows.append({"descriptor": name, "importance": np.mean(drops)})
    return pd.DataFrame(rows).sort_values("importance", ascending=False)


# ============ bp (끓는점) ============
print("=== bp elbow curve ===")
t0 = time.time()
bp = pd.read_csv("./data/processed/crc_bp_c.csv")
bp_fp = np.load("./data/processed/crc_bp_c_fingerprint.npy")
y_bp = bp["Processed tbp/ºC"].values

desc_names, bp_all_desc = compute_all_descriptors(bp["SMILES"])
FP_DIM = bp_fp.shape[1]

X_tr_fp, X_te_fp, desc_tr, desc_te, y_tr, y_te = train_test_split(bp_fp, bp_all_desc, y_bp, test_size=0.2, random_state=42)
X_full_tr = np.hstack([X_tr_fp, desc_tr])
X_full_te = np.hstack([X_te_fp, desc_te])

full_model = HistGradientBoostingRegressor(random_state=42)
full_model.fit(X_full_tr, y_tr)
bp_importance = permutation_importance_train_only(full_model, X_full_tr, y_tr, FP_DIM, desc_names, r2_score)
bp_importance.to_csv("./models/bp_descriptor_importance.csv", index=False)

print(f"{'N':>5} {'R2':>10}")
for n in N_CANDIDATES:
    top_names = bp_importance.head(n)["descriptor"].tolist() if n > 0 else []
    idxs = [desc_names.index(nm) for nm in top_names]
    X_tr = np.hstack([X_tr_fp, desc_tr[:, idxs]]) if n > 0 else X_tr_fp
    X_te = np.hstack([X_te_fp, desc_te[:, idxs]]) if n > 0 else X_te_fp
    m = HistGradientBoostingRegressor(random_state=42)
    m.fit(X_tr, y_tr)
    print(f"{n:>5} {r2_score(y_te, m.predict(X_te)):>10.4f}")
print(f"bp 완료 ({time.time()-t0:.0f}초)\n")

# ============ 유해성 (Tox21, 12개 태스크) ============
print("=== 유해성(Tox21) elbow curve (12개 태스크 평균 AUC) ===")
t0 = time.time()
tox21 = pd.read_csv("./data/hazard/moleculenet_tox21.csv")
tox21_fp = np.load("./data/processed/moleculenet_tox21_fingerprint.npy")
label_cols = [c for c in tox21.columns if c not in ("SMILES", "InChIKey")]

desc_names_tox, tox21_all_desc = compute_all_descriptors(tox21["SMILES"])
FP_DIM = tox21_fp.shape[1]

N_CANDIDATES_CLF = [0, 10, 20, 30, 40, 60, 100, 217]  # 분류 12개라 개수 줄임
auc_by_n = {n: [] for n in N_CANDIDATES_CLF}
per_task_importance = {}

for task in label_cols:
    mask = tox21[task].notna()
    fp_t = tox21_fp[mask.values]
    desc_t = tox21_all_desc[mask.values]
    y_t = tox21.loc[mask, task].values

    X_tr_fp, X_te_fp, desc_tr, desc_te, y_tr, y_te = train_test_split(
        fp_t, desc_t, y_t, test_size=0.2, stratify=y_t, random_state=42
    )
    X_full_tr = np.hstack([X_tr_fp, desc_tr])
    full_model = RandomForestClassifier(class_weight="balanced", n_estimators=300, random_state=42)
    full_model.fit(X_full_tr, y_tr)
    imp = permutation_importance_train_only(full_model, X_full_tr, y_tr, FP_DIM, desc_names_tox, roc_auc_score)
    per_task_importance[task] = imp

    for n in N_CANDIDATES_CLF:
        top_names = imp.head(n)["descriptor"].tolist() if n > 0 else []
        idxs = [desc_names_tox.index(nm) for nm in top_names]
        X_tr = np.hstack([X_tr_fp, desc_tr[:, idxs]]) if n > 0 else X_tr_fp
        X_te = np.hstack([X_te_fp, desc_te[:, idxs]]) if n > 0 else X_te_fp
        m = RandomForestClassifier(class_weight="balanced", n_estimators=300, random_state=42)
        m.fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])
        auc_by_n[n].append(auc)
    print(f"  {task} 완료 ({time.time()-t0:.0f}초 누적)")

print(f"\n{'N':>5} {'평균 AUC':>10}")
for n in N_CANDIDATES_CLF:
    print(f"{n:>5} {np.mean(auc_by_n[n]):>10.4f}")
print(f"유해성 완료 ({time.time()-t0:.0f}초)")

=== bp elbow curve ===
    N         R2
    0     0.7213
    5     0.9230
   10     0.9237
   15     0.9227
   20     0.9231
   25     0.9258
   30     0.9264
   40     0.9302
   60     0.9288
  100     0.9288
  217     0.9287
bp 완료 (98초)

=== 유해성(Tox21) elbow curve (12개 태스크 평균 AUC) ===


  NR-AR 완료 (241초 누적)
  NR-AR-LBD 완료 (423초 누적)
  NR-AhR 완료 (629초 누적)
  NR-Aromatase 완료 (790초 누적)
  NR-ER 완료 (1029초 누적)
  NR-ER-LBD 완료 (1221초 누적)
  NR-PPAR-gamma 완료 (1372초 누적)
  SR-ARE 완료 (1563초 누적)
  SR-ATAD5 완료 (1742초 누적)
  SR-HSE 완료 (1922초 누적)
  SR-MMP 완료 (2097초 누적)
  SR-p53 완료 (2288초 누적)

    N     평균 AUC
    0     0.8251
   10     0.8461
   20     0.8464
   30     0.8487
   40     0.8514
   60     0.8477
  100     0.8506
  217     0.8472
유해성 완료 (2288초)
